In [10]:
import pandas as pd
import numpy as np
from functools import reduce

In [20]:
path_1 = "../splits/libre_int_1_mins/data_filtered_manually_gl_stats.csv"
path_5 = "../splits/libre_int_5_mins/data_filtered_manually_gl_stats.csv"
path_15 = "../splits/lunch_dinner/filter_before_60_after_120.csv"

In [21]:
df_1 = pd.read_csv(path_1)
df_5 = pd.read_csv(path_5)
df_15 = pd.read_csv(path_15)

df_1.shape, df_5.shape, df_15.shape

((457, 47), (457, 47), (451, 47))

In [22]:
key = "Image path"
cols = ["AUC", "iAUC"]

In [23]:
df15_small = df_15[[key] + cols].rename(columns={
    "AUC": "AUC_15",
    "iAUC": "iAUC_15"
})

df5_small = df_5[[key] + cols].rename(columns={
    "AUC": "AUC_5",
    "iAUC": "iAUC_5"
})

df1_small = df_1[[key] + cols].rename(columns={
    "AUC": "AUC_1",
    "iAUC": "iAUC_1"
})

# Merge all 3 datasets
cmp = reduce(
    lambda left, right: left.merge(right, on=key, how="outer"),
    [df15_small, df5_small, df1_small]
)

In [24]:
print("Total rows:", len(cmp))

print("Missing 15-min:", cmp["AUC_15"].isna().sum())
print("Missing 5-min:", cmp["AUC_5"].isna().sum())
print("Missing 1-min:", cmp["AUC_1"].isna().sum())

Total rows: 457
Missing 15-min: 6
Missing 5-min: 0
Missing 1-min: 0


In [25]:
cmp.columns

Index(['Image path', 'AUC_15', 'iAUC_15', 'AUC_5', 'iAUC_5', 'AUC_1',
       'iAUC_1'],
      dtype='object')

In [26]:
cmp_common = cmp.dropna(subset=[
    "AUC_15", "AUC_5", "AUC_1",
    "iAUC_15", "iAUC_5", "iAUC_1"
]).copy()

In [27]:
pairs = [("15", "5"), ("15", "1"), ("5", "1")]

for col in ["AUC", "iAUC"]:
    for a, b in pairs:
        cmp_common[f"{col}_diff_{a}_vs_{b}"] = cmp_common[f"{col}_{b}"] - cmp_common[f"{col}_{a}"]
        cmp_common[f"{col}_abs_diff_{a}_vs_{b}"] = cmp_common[f"{col}_diff_{a}_vs_{b}"].abs()

        cmp_common[f"{col}_rel_diff_%_{a}_vs_{b}"] = np.where(
            cmp_common[f"{col}_{a}"].abs() > 1e-8,
            100 * cmp_common[f"{col}_diff_{a}_vs_{b}"] / cmp_common[f"{col}_{a}"],
            np.nan
        )

In [28]:
for col in ["AUC", "iAUC"]:
    print(f"\n===== {col} =====")

    for a, b in pairs:
        diff_col = f"{col}_diff_{a}_vs_{b}"
        abs_col = f"{col}_abs_diff_{a}_vs_{b}"

        print(f"\n{a} min vs {b} min")
        print("Mean diff:", cmp_common[diff_col].mean())
        print("Mean abs diff:", cmp_common[abs_col].mean())
        print("Median abs diff:", cmp_common[abs_col].median())
        print("Max abs diff:", cmp_common[abs_col].max())
        print("Correlation:", cmp_common[[f"{col}_{a}", f"{col}_{b}"]].corr().iloc[0, 1])


===== AUC =====

15 min vs 5 min
Mean diff: 4.045824094604387
Mean abs diff: 10.066518847006458
Median abs diff: 6.666666666664241
Max abs diff: 111.66666666666424
Correlation: 0.9999935174316141

15 min vs 1 min
Mean diff: 4.566888396156645
Mean abs diff: 11.377827050997793
Median abs diff: 7.200000000000728
Max abs diff: 125.0666666666657
Correlation: 0.9999920643038742

5 min vs 1 min
Mean diff: 0.5210643015522577
Mean abs diff: 1.311308203991362
Median abs diff: 0.8000000000010914
Max abs diff: 13.400000000001455
Correlation: 0.9999998878928926

===== iAUC =====

15 min vs 5 min
Mean diff: 11.149470545083238
Mean abs diff: 13.020425224450632
Median abs diff: 7.9999999999990905
Max abs diff: 185.51945783217752
Correlation: 0.9999709516710742

15 min vs 1 min
Mean diff: 12.84248840713028
Mean abs diff: 14.859524568940328
Median abs diff: 9.399999999999409
Max abs diff: 211.31945783217753
Correlation: 0.9999642854891053

5 min vs 1 min
Mean diff: 1.6930178620470377
Mean abs diff: 1.8

In [29]:
cmp_common.sort_values("iAUC_abs_diff_15_vs_1", ascending=False)[
    [
        key,
        "iAUC_15", "iAUC_5", "iAUC_1",
        "iAUC_diff_15_vs_1",
        "iAUC_abs_diff_15_vs_1",
        "iAUC_rel_diff_%_15_vs_1"
    ]
].head(20)

,Image path,iAUC_15,iAUC_5,iAUC_1,iAUC_diff_15_vs_1,iAUC_abs_diff_15_vs_1,iAUC_rel_diff_%_15_vs_1
82,photos/00000029-PHOTO-2023-11-7-12-18-0.jpg,110.309533,295.828991,321.628991,211.319458,211.319458,191.569533
353,photos/00000070-PHOTO-2022-4-24-13-7-0.jpg,5658.322581,5773.400000,5787.200000,128.877419,128.877419,2.277661
409,photos/00000076-PHOTO-2025-10-18-19-15-0.jpg,573.916237,668.263990,675.962868,102.046631,102.046631,17.780753
276,photos/00000040-PHOTO-2021-3-30-16-9-0.jpg,6384.500000,6466.166667,6475.966667,91.466667,91.466667,1.432636
321,photos/00000026-PHOTO-2024-1-20-17-19-0.jpg,2082.476398,2157.775000,2164.775000,82.298602,82.298602,3.951958
263,photos/00000023-PHOTO-2021-5-7-13-43-0.jpg,8409.000000,8484.000000,8490.000000,81.000000,81.000000,0.963254
2,photos/00000016-PHOTO-2020-5-2-19-52-0.jpg,134.166776,195.316631,202.716631,68.549854,68.549854,51.093017
160,photos/00000007-PHOTO-2024-1-25-17-51-0.jpg,516.111628,576.789744,581.723077,65.611449,65.611449,12.712647
345,photos/00000032-PHOTO-2022-1-8-12-18-0.jpg,347.987526,408.411404,413.344737,65.357211,65.357211,18.781481
225,photos/00000063-PHOTO-2024-2-14-13-32-0.jpg,19.768519,83.250000,83.250000,63.481481,63.481481,321.124122


In [30]:
duration = 120

for col in ["AUC", "iAUC"]:
    for a, b in [("15", "5"), ("15", "1"), ("5", "1")]:
        abs_col = f"{col}_abs_diff_{a}_vs_{b}"
        avg_col = f"{col}_avg_abs_diff_per_min_{a}_vs_{b}"
        cmp_common[avg_col] = cmp_common[abs_col] / duration

        print(f"\n{col} {a} vs {b}")
        print(cmp_common[avg_col].describe())


AUC 15 vs 5
count    451.000000
mean       0.083888
std        0.102452
min        0.000000
25%        0.018056
50%        0.055556
75%        0.111111
max        0.930556
Name: AUC_avg_abs_diff_per_min_15_vs_5, dtype: float64

AUC 15 vs 1
count    451.000000
mean       0.094815
std        0.111909
min        0.000000
25%        0.023889
50%        0.060000
75%        0.123333
max        1.042222
Name: AUC_avg_abs_diff_per_min_15_vs_1, dtype: float64

AUC 5 vs 1
count    451.000000
mean       0.010928
std        0.013256
min        0.000000
25%        0.001667
50%        0.006667
75%        0.015000
max        0.111667
Name: AUC_avg_abs_diff_per_min_5_vs_1, dtype: float64

iAUC 15 vs 5
count    451.000000
mean       0.108504
std        0.140266
min        0.000000
25%        0.027778
50%        0.066667
75%        0.138889
max        1.545995
Name: iAUC_avg_abs_diff_per_min_15_vs_5, dtype: float64

iAUC 15 vs 1
count    451.000000
mean       0.123829
std        0.155437
min        0.0

In [31]:
for col in ["AUC", "iAUC"]:
    for a, b in [("15", "5"), ("15", "1"), ("5", "1")]:
        rel_col = f"{col}_rel_diff_%_{a}_vs_{b}"
        print(f"\n{col} {a} vs {b}")
        print(cmp_common[rel_col].describe())


AUC 15 vs 5
count    451.000000
mean       0.025251
std        0.120708
min       -0.896861
25%       -0.024391
50%        0.015935
75%        0.077443
max        0.605653
Name: AUC_rel_diff_%_15_vs_5, dtype: float64

AUC 15 vs 1
count    451.000000
mean       0.028690
std        0.133463
min       -1.004484
25%       -0.028602
50%        0.020747
75%        0.088397
max        0.627504
Name: AUC_rel_diff_%_15_vs_1, dtype: float64

AUC 5 vs 1
count    451.000000
mean       0.003423
std        0.015739
min       -0.108597
25%       -0.002568
50%        0.000811
75%        0.010077
max        0.078087
Name: AUC_rel_diff_%_5_vs_1, dtype: float64

iAUC 15 vs 5
count    451.000000
mean       5.065501
std       25.770002
min      -38.953488
25%        0.062343
50%        0.316957
75%        1.138369
max      321.124122
Name: iAUC_rel_diff_%_15_vs_5, dtype: float64

iAUC 15 vs 1
count    451.000000
mean       6.005134
std       28.074594
min      -18.604651
25%        0.084359
50%        0.3